In [8]:
%pip install google-play-scraper
%pip install Sastrawi

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.7/209.7 kB 4.9 MB/s eta 0:00:0000:01


# IMPORT LIBRARY

In [9]:
from google_play_scraper import app, reviews, Sort, reviews_all

import pandas as pd
pd.options.mode.chained_assignment = None
import numpy as np
seed = 0
np.random.seed(seed) # Mengatur seed untuk reproduktibilitas
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score

import datetime as dt
import re
import string
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

from Sastrawi.Stemmer.StemmerFactory import StemmerFactory # Stemming (penghilangan imbuhan kata) dalam bahasa Indonesia
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory # Stopword removal (penghilangan kata-kata umum yang tidak memiliki makna penting) dalam bahasa Indonesia

from wordcloud import WordCloud
import nltk
nltk.download('punkt_tab')
nltk.download('stopwords')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

# Scraping Dataset

In [ ]:
# Mengimpor pustaka google_play_scraper untuk mengakses ulasan dan informasi aplikasi dari Google Play Store.
from google_play_scraper import app, reviews_all, Sort

# Mengambil semua ulasan dari aplikasi dengan ID 'com.byu.id' di Google Play Store.
# Proses scraping mungkin memerlukan beberapa saat tergantung pada jumlah ulasan yang ada.

scrapview = reviews_all(
    'com.byu.id', # ID aplikasi
    lang = 'id', #Bahasa ulasan
    country = 'id', #Negara
    sort = Sort.MOST_RELEVANT, #Urutan ulasan berdasarkan relevansi
    count = 1000 #Jumlah ulasan yang diambil
)
scrapview

In [ ]:
import csv

with open('ulasan_aplikasi.csv', 'w', newline='', encoding='utf-8') as file:
    writer = csv.writer(file)
    writer.writerow(['Review'])
    for review in scrapview:
        writer.writerow([review['content']])

# Loading Dataset
Selanjutnya, jika sudah berhasil, kita dapat melihat dataset tersebut dengan kode seperti berikut.



In [ ]:
app_reviews_df = pd.DataFrame(scrapview)
app_reviews_df.head()

In [ ]:
app_reviews_df = pd.DataFrame(scrapview)
jumlah_ulasan, jumlah_kolom = app_reviews_df.shape

app_reviews_df

In [ ]:
app_reviews_df.info()

In [ ]:
clean_df  = app_reviews_df.dropna()
clean_df = clean_df.drop_duplicates()

In [ ]:
clean_df.info()

In [ ]:
import re
import string
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

def cleaningText(text):
    text = re.sub(r'@[A-Za-z0-9]+', '', text) #menghapus mention
    text = re.sub(r'#[A-Za-z0-9]+', '', text) #menghapus hastag
    text = re.sub(r'RT[\s]', '', text) # menghapus RT
    text = re.sub(r"http\S+", '', text) #menghapus URL
    text = re.sub(r'[0-9]+', '', text) # menghapus angka
    text = re.sub(r'[^\w\s]', '', text) # menghapus karakter selain huruf dan angka

    text = text.replace('\n', ' ') #menganti garis baru dengan spasi
    text = text.translate(str.maketrans('', '', string.punctuation)) #menghapus tanda baca
    text = text.strip(' ')      #menghapus spasi di awal dan akhir teks
    return text

def casefoldingText(text): 
    text = text.lower() #mengubah semua huruf menjadi huruf kecil
    return text

def tokenizingText(text):
    teks = word_tokenize(text)
    return teks

def filteringText(text):
    listStopWords = set(stopwords.words('indonesian')) #mengambil daftar stopword bahasa Indonesia
    listStopWords = set(stopwords.words('english'))
    listStopWords.update(['yg', 'dg', 'rt', 'dgn', 'ny', 'klo', 'kalo', 'amp', 'biar', 'bikin', 'bilang', 'gak', 'ga', 'krn', 'nya', 'nih', 'sih', 'si', 'tau', 'tuh', 'utk', 'ya', 'jd', 'jdi'])
    filtered = []
    for txt in text:
        if txt not in listStopWords:
            filtered.append(txt)
    text = filtered
    return text

def stemmingText(text): # Mengurangi kata ke bentuk dasarnya
    # Membuat objek stemmer
    factory = StemmerFactory()
    stemmer = factory.create_stemmer()

    #memecah teks menjadi kata-kata
    words = text.split()
    stemmed_words = [stemmer.stem(word) for word in words]
    stemmed_text = ' '.join(stemmed_words)
    return stemmed_text

def toSentence(text):
    sentence =' '.join(word for word in text)
    return sentence

In [ ]:
slangwords = {"@": "di", "abis": "habis", "wtb": "beli", "masi": "masih", "wts": "jual", "wtt": "tukar", "bgt": "banget", "maks": "maksimal"}
def fix_slangwords(text):
    words = text.split()
    fixed_words = []
 
    for word in words:
        if word.lower() in slangwords:
            fixed_words.append(slangwords[word.lower()])
        else:
            fixed_words.append(word)
 
    fixed_text = ' '.join(fixed_words)
    return fixed_text

In [ ]:
# Membersihkan teks dan menyimpannya di kolom 'text_clean'
clean_df['text_clean'] = clean_df['content'].apply(cleaningText)
 
# Mengubah huruf dalam teks menjadi huruf kecil dan menyimpannya di 'text_casefoldingText'
clean_df['text_casefoldingText'] = clean_df['text_clean'].apply(casefoldingText)
 
# Mengganti kata-kata slang dengan kata-kata standar dan menyimpannya di 'text_slangwords'
clean_df['text_slangwords'] = clean_df['text_casefoldingText'].apply(fix_slangwords)
 
# Memecah teks menjadi token (kata-kata) dan menyimpannya di 'text_tokenizingText'
clean_df['text_tokenizingText'] = clean_df['text_slangwords'].apply(tokenizingText)
 
# Menghapus kata-kata stop (kata-kata umum) dan menyimpannya di 'text_stopword'
clean_df['text_stopword'] = clean_df['text_tokenizingText'].apply(filteringText)
 
# Menggabungkan token-token menjadi kalimat dan menyimpannya di 'text_akhir'
clean_df['text_akhir'] = clean_df['text_stopword'].apply(toSentence)

In [ ]:
clean_df

# PELABELAN
Sebelum masuk ke tahap pemodelan, langkah yang dilakukan adalah pelabelan. Pelabelan adalah proses pemberian kategori atau label pada setiap entri data berdasarkan informasi yang tersedia. Dalam konteks ini, setiap entri dataset diberikan label sentimen berdasarkan analisis teksnya. Dengan demikian, tahapan pelabelan menjadi dasar untuk proses selanjutnya dalam membangun model klasifikasi sentimen.

In [ ]:
import csv 
import requests
from io import StringIO

lexicon_positive = dict()
response = requests.get(('https://raw.githubusercontent.com/angelmetanosaa/dataset/main/lexicon_positive.csv'))

if response.status_code == 200:
     # Membaca teks respons sebagai file CSV menggunakan pembaca CSV dengan pemisah koma
    reader = csv.reader(StringIO(response.text), delimiter=',')
    for row in reader :
         # Menambahkan kata-kata positif dan skornya ke dalam kamus lexicon_positive
        lexicon_positive[row[0]] = int(row[1])

else :
    print("Failed to fetch positive lexicon data")
        
lexicon_negative = dict()
response = requests.get('https://raw.githubusercontent.com/angelmetanosaa/dataset/main/lexicon_negative.csv')

if response.status_code == 200:
    reader = csv.reader(StringIO(response.text), delimiter=',')
    for row in reader :
        lexicon_negative[row[0]] = int(row[1])

else : 
    print("Failed to fetch negative lexicon data")

In [ ]:
def sentiment_analysis_lexicon_indonesia(text):
    #inisialisasi skor sentimen
    score = 0

    for word in text:
        if word in lexicon_positive:
             # Jika kata ada dalam kamus positif, tambahkan skornya ke skor sentimen
             score = score + lexicon_positive[word]
    
    for word in text:
        if word in lexicon_negative:
            # Jika kata ada dalam kamus negatif, tambahkan skornya ke skor sentimen
            score = score + lexicon_negative[word]
        
    polarity = ''

    if score > 0:
        polarity = 'positive'
    elif score < 0:
        polarity = 'negative'
    else:
        polarity = 'neutral'

    return score, polarity

In [ ]:
results = clean_df['text_stopword'].apply(sentiment_analysis_lexicon_indonesia)
results = list(zip(*results))
clean_df['polarity_score'] = results[0]
clean_df['polarity'] = results[1]
print(clean_df['polarity'].value_counts())

In [ ]:
clean_df

In [ ]:
plt.pie(clean_df['polarity'].value_counts(), labels=clean_df['polarity'].value_counts().index, autopct='%1.1f%%')
plt.title('Distribusi Sentimen Ulasan Aplikasi By.U')
plt.show()

In [ ]:
from wordcloud import WordCloud, STOPWORDS
from nltk.corpus import stopwords as nltk_stopwords

stop_id = set(nltk_stopwords.words('indonesian'))
stopwords = set(STOPWORDS)
stopwords.update(['dan', 'di', 'bisa', 'tidak', 'lagi', 'juga', 'kalau', 'ini', 
                  'itu', 'untuk', 'dengan', 'saya', 'byu', 'aplikasi', 'pake', 
                  'udah', 'gak', 'ga', 'kalo', 'aja', 'dari', 'ke'])
final_stopwords = stop_id.union(stopwords)

all_text = ' '.join(clean_df['text_akhir'])

wc = WordCloud(background_color='white', width=800, height=400, stopwords=final_stopwords)
wc.generate(all_text)

plt.figure(figsize=(10, 5))
plt.imshow(wc, interpolation='bilinear')
plt.axis('off')
plt.title('Word Cloud Semua Ulasan By.U', fontsize=20)
plt.show()